# មេរៀនទី 12 - ការកាត់បន្ថយប្រវត្តិការជជែកជាមួយ Agent Scratchpad

កំណត់ត្រានេះបង្ហាញពីវិធីគ្រប់គ្រងបរិបទក្នុងការសន្ទនាយ៉ាងវែងដោយប្រើ Microsoft Agent Framework។ នៅពេលដែលការសន្ទនាឡើងខ្លាំង ចំនួន token កើនឡើង - ផុតកំណត់បរិបទរបស់ម៉ូដែល។ យើងដោះស្រាយបញ្ហានេះជាមួយ **លំនាំសង្ខេបបរិបទ** និង **agent scratchpad** សម្រាប់ចងចាំអចិន្ត្រៃយ៍។

## អ្វីដែលអ្នកនឹងរៀន៖
1. **ហេតុអ្វីបានជា ការគ្រប់គ្រងបរិបទមានសារៈសំខាន់**៖ ការយល់ដឹងពីកំណត់ token និងបង្អួចបរិបទ
2. **Agents ដែលមានការ​យល់ដឹងពីបរិបទ**៖ ការបង្កើត agent ដែលគ្រប់គ្រងបរិបទជជែករបស់ខ្លួន
3. **លំនាំសង្ខេបបរិបទ**៖ ការប្រើឧបករណ៍ដើម្បីសង្ខេបប្រវត្តិការជជែក
4. **Agent Scratchpad**៖ ចងចាំអចិន្ត្រៃយ៍ដែលនៅរស់រវើកក្រោយការកាត់បន្ថយបរិបទ

## ការត្រៀមខ្លួន៖
- ការតំឡើង Azure OpenAI ជាមួយអថេរបរិស្ថានដែលត្រូវបានកំណត់
- ការយល់ដឹងពីមូលដ្ឋានគំនិត agent ពីមេរៀនមុនៗ


## ការតំឡើង


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## ហេតុអ្វីបានជា ការគ្រប់គ្រងបរិបទមានសារៈសំខាន់

រាល់ LLM ទាំងអស់មាន **បង្អួចបរិបទ** កំណត់មួយ — ចំនួនអក្សរពុំអាចលើសដែលវាអាចដំណើរការបានក្នុងសំណើតែមួយ។ នៅពេលការពិភាក្សាច្រើនជំហានកើតឡើង៖

- **ចំនួន token កើនឡើងដោយបន្ទាត់** ជាមួយនឹងសារ​របស់អ្នកប្រើ និងចម្លើយរបស់ជំនួយការ។
- **token សម្រាប់ prompt មានតម្លៃខ្ពស់** ព្រោះប្រវត្តិទាំងមូលត្រូវបានបញ្ជូនវិញរៀងរាល់ជំហាន។
- ចុងក្រោយ ការពិភាក្សា **លើសព្រំដែនបង្អួចបរិបទ** ហើយម៉ូដែលនឹងកាត់បន្ថយ ឬក៏បញ្ហាដោយកំហុស។

### វិធីសាស្ត្រសម្រាប់គ្រប់គ្រងបរិបទ

| វិធីសាស្ត្រ | វាដំណើរការ​ដូចម្តេច | ផលប៉ះពាល់ |
|---|---|---|
| **ការកាត់បន្ថយ** | ដកសារ​ចាស់បំផុត | រាល់បរិបទដើមបាត់បង់ |
| **ការសង្ខេប** | ប្រមួលសារ​ចាស់ជាសេចក្ដីសង្ខេប | មានពត៌មានមួយចំនួនបាត់បង់ ប៉ុន្តែចំណុចសំខាន់នៅតែរក្សា |
| **Scratchpad / ចងចាំខាងក្រៅ** | រក្សាទុកពត៌មានសំខាន់ខាងក្រៅការពិភាក្សា | ត្រូវការហៅឧបករណ៍ ប៉ុន្តែធន់ទ្រាំ​នឹង​ការកាត់បន្ថយណាមួយ |

ក្នុងកំណត់ត្រានេះ យើងផ្សំតួរវិជ្ជាជីវៈ **ការសង្ខេប** ជាមួយ **ឧបករណ៍ scratchpad** ដើម្បីឱ្យភ្នាក់ងារអាចរក្សាសភាពបន្តបន្ទាត់បាន ទាំងពេលប្រវត្តិការពិភាក្សាត្រូវបានសង្ខេប។


## បង្កើតភ្នាក់ងារដែលមានការយល់ដឹងអំពីបរិបទ


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## ការស្ទាក់ស្ទើរ នៃ ការស្តាប់សន្ទនាផ្សេងៗយូរ

យើងមកដំណើរការ សន្ទនាច្រើនជំហាន ដើម្បីមើលថា ព័ត៌មានបរិបទ ត្រូវបានស្ដារឡើងវិញយ៉ាងដូចម្តេច។ អ្នកលក់គួរតែរក្សាទុកព័ត៌មានសំខាន់ៗ (ចំណង់ចំណូលចិត្ត, ថវិកា, កាលបរិច្ឆេទធ្វើដំណើរ) តាមរយៈជំហានៗ ហើយបង្ហាញភាពបន្តតំណាងឲ្យការតភ្ជាប់។ 


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

សូមប្រុងប្រយ័ត្នថាភ្នាក់ងារនេះរក្សាបរិបទពីជើងខ្លួនកន្លងមក — វាស្គាល់ពីជប៉ុន, ស៊ូស៊ី, មណ្ឌល, ថតរូប,ថវិកា $3000, ការធ្វើដំណើរតែម្នាក់ឯង និងឱកាសខែមេសា។ ក្នុងការសន្ទនាគ្រាប់ខ្លីនេះវាដំណើរការល្អ ប៉ុន្តែពេលដែលការសន្ទនាត្រូវបន្ត ប្រវត្តិពេញលេញនឹងមានថ្លៃតម្លៃខ្ពស់ក្នុងការផ្ញើឡើងវិញ។

យើងសូមបន្តការសន្ទនាជាមួយជើងខ្លួនបន្ថែមទៀតដើម្បីមើលការបន្ថែមបរិបទ៖


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## គំរូការសង្ខេបបរិបទ

នៅពេលសន្ទនាបន្ថែម ចំពោះយើងអាចប្រើប្រាស់ **ឧបករណ៍សង្ខេប** ដើម្បីដាក់បរិបទដែលបានបូកចូលទៅជារូបមន្តសង្ខេបមួយ។ អ្នកចាត់ការប្រើឧបករណ៍នេះដើម្បីកត់ត្រាចំណង់ចំណូលចិត្តសំខាន់ៗ ដូច្នេះបើសារចាស់ៗត្រូវបានលុបក៏ដោយ លើសពីនេះព័ត៌មានសំខាន់ៗនៅតែត្រូវបានរក្សាទុក។

គំរូនេះគឺជាគ្រឹះសម្រាប់ការកាត់បន្ថយប្រវត្តិលម្អិតជាងនេះទៀត៖
1. អ្នកចាត់ការទទួលស្គាល់ប 사실សំខាន់ៗពីការសន្ទនា
2. វាហៅឧបករណ៍សង្ខេបដើម្បីរក្សាទុកពួកវា
3. សារចាស់ៗអាចត្រូវបានលុបដោយសុវត្ថិភាព ព្រោះសង្ខេបបានចាប់យកអ្វីដែលសំខាន់ៗ

ខាងក្រោមយើងកំណត់ឧបករណ៍ `summarize_preferences` ដែលអាចត្រូវបានអ្នកចាត់ការហៅដើម្បីកត់ត្រាសង្ខេបគ្រប់បរិបទដែលបានរៀន។


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## សេចក្តីសង្ខេប

ក្នុងមេរៀននេះ អ្នកបានរៀនពីរបៀបគ្រប់គ្រងបរិបទក្នុងសន្ទនារយៈពេលវែងរបស់ភ្នាក់ងារដោយប្រើ Microsoft Agent Framework ៖

### គំនិតសំខាន់ៗ
- **បង្អួចបរិបទមានកំណត់** — រាល់តួអក្សរនៅក្នុងប្រវត្តិសន្ទនាគឺមានតម្លៃប្រាក់ និងរាប់ចូលក្នុងដែនកំណត់។
- **ឧបករណ៍សង្ខេប** អនុញ្ញាតឲ្យភ្នាក់ងារប្រមូលបរិបទដែលបានរក្សាទុកជាសង្ខេបដើម្បីកាត់បន្ថយការប្រើតួអក្សរជាមួយនិងរក្សាទុកព័ត៌មានសំខាន់។
- **ក្រដាសសរសេររបស់ភ្នាក់ងារ** ផ្ដល់នូវអង្គចងចាំខាងក្រៅដែលនឹងនៅរួចរាល់ ទោះបីត្រូវបន្ថយសន្ទនាដែរ។

### អ្វីដែលអ្នកបានសាងសង់
- **ភ្នាក់ងារដែលមានការយល់ដឹងពីបរិបទ** ដែលថែរក្សាទំនាក់ទំនងតាមរយៈសន្ទនាច្រើនជុំ
- **ឧបករណ៍សង្ខេប** (`summarize_preferences`) ដែលកត់ត្រាពត៌មានសំខាន់របស់អ្នកប្រើប្រាស់ក្នុងទ្រង់ទ្រាយខ្លីសង្ខេប
- **សន្ទនាច្រើនជុំ** ដែលបង្ហាញពីការរក្សាទុកបរិបទនិងការដោះស្រាយការផ្លាស់ប្តូរ

### កម្មវិធីក្នុងពិភពពិត
- **រសជាតិបម្រើអតិថិជន**៖ រំលឹកចំណង់ចំណូលចិត្តក្នុងការបម្រើអ្នកយូរអង្វែង
- **ជំនួយផ្ទាល់ខ្លួន**៖ តាមដានគម្រោងកំពុងដំណើរការដោយគ្មានការពន្យល់បរិបទឡើងវិញ
- **គ្រូបង្រៀន**៖ រក្សាទុកលទ្ធផលសិស្សនៅក្នុងការប្រាស្រ័យទាក់ទងជាច្រើន

### ជំហានបន្ទាប់
- អនុវត្តឧបករណ៍ scratchpad ពេញលេញជាមួយការរក្សាទុកនៅលើឯកសារ
- បន្ថែមការកាត់បន្ថយប្រវត្តិជាស្វ័យប្រវត្តិបន្ទាប់ពីការសង្ខេប
- ប្រមូលផ្តុំជាមួយមូលដ្ឋានទិន្នន័យវ៉ិចទ័រសម្រាប់ស្វែងរកអង្គចងចាំមានអត្ថន័យ
- បង្កើតភ្នាក់ងារដែលអាចបន្តសន្ទនាបានបន្តពីរបៀបពេញលេញនៅពេលក្រោយ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
